# Chapter 7: Log-Structured Storage
*From: Database Internals*

## TL;DR

- **LSM Trees** buffer writes in an in-memory **memtable** (backed by a WAL), then flush sorted **immutable SSTables** to disk. Random writes become sequential; random reads become a merge across many files.
- **Tombstones** encode deletes as records. Lookups multiway-merge iterators from memtable + SSTables through a min-heap; reconciliation keeps the newest timestamp and respects tombstone shadowing.
- **Compaction** rewrites tables to bound read amplification and reclaim space. Leveled (RocksDB) partitions data into exponentially-sized non-overlapping levels; size-tiered (Cassandra) groups tables by size; time-window drops whole files for TTL workloads.
- The **RUM Conjecture** says you can optimize at most two of Read / Update / Memory overhead. LSM explicitly trades read, write, and space amplification — aggressive compaction lowers the first two and raises the third.
- **Bloom filters** let lookups skip SSTables that definitely don't contain the key; **skiplists** give memtables O(log n) probabilistic ops without rotations.
- **Unordered** variants (Bitcask, WiscKey) skip sorting on disk to slash compaction cost — Bitcask keeps all keys in RAM and does point-only queries; WiscKey keeps the LSM over keys but spills values into an unsorted vLog.
- **Log stacking** (app LSM on FS log on FTL) creates duplicated GC, misaligned segments, and interleaved write streams. LLAMA fixes it by making the log access-method aware; Open-Channel SSDs let the app bypass the FTL entirely.

## The Problem

Classical B-Trees do in-place page rewrites. On an SSD every small logical update becomes a full-page physical write, splits propagate rewrites up the tree, and pages reserve slack for future inserts (space amplification). The chapter frames the alternative this way: *"Log-structured merge trees take their name from log-structured filesystems, which write all modifications on disk in a log-like file... Keeping data files immutable favors sequential writes: data is written on the disk in a single pass and files are append-only."*

The core bet is simple: accept extra work at read time (merge + reconcile across several files) and during background compaction, in exchange for turning every write into a pure append. That trade is good for SSDs, which love sequential writes and hate in-place rewrites, and it is good for high-ingest workloads where writes dominate reads.

But the trade is not free. The chapter names the three taxes precisely:

- **Read amplification** — a point lookup may have to probe the memtable plus several SSTables across several levels.
- **Write amplification** — compaction rewrites records many times as they migrate from low levels to the bottommost level.
- **Space amplification** — multiple versions of the same key coexist until compaction reconciles them.

Every design choice in this chapter — compaction strategy, Bloom filter sizing, key/value separation, log stacking — is an attempt to shift one of those taxes at the cost of another. The RUM Conjecture generalizes this: you can pick at most two of Read, Update, and Memory overhead to optimize.

## Amplification Math

Before taxonomizing LSM design choices, let's quantify what we are fighting. A realistic production LSM (RocksDB-style, leveled) has a 64 MB memtable, ~10x fanout per level, 10 levels, and a 1% Bloom filter false-positive rate. Here are write, read, and space amplification for a leveled vs. size-tiered scheme.

In [ ]:
# LSM amplification math: write, read, space under two compaction strategies.
import math

MEMTABLE_MB     = 64            # size of one memtable flush
FANOUT          = 10            # per-level size ratio
NUM_LEVELS      = 7             # L0 .. L6
RECORDS         = 1_000_000_000 # total records in the tree
RECORD_BYTES    = 100           # avg key+value+overhead
BLOOM_FPR       = 0.01          # 1% false-positive rate, standard
TABLES_PER_TIER = 4             # size-tiered groups 4 tables per size class

# --- Write amplification -------------------------------------------------
# Leveled: every byte written at L0 eventually migrates down to L_{n-1};
# on each merge into a deeper level, it is rewritten once with the deeper
# level's data. The amortized write amp is roughly FANOUT per level.
leveled_WA = 1 + (NUM_LEVELS - 1) * FANOUT   # +1 for the initial memtable flush

# Size-tiered: each table is rewritten once per tier promotion; a record
# lives through roughly log_{k}(N) tier promotions, rewritten in full each time.
tiered_levels = math.ceil(math.log(RECORDS * RECORD_BYTES /
                                   (MEMTABLE_MB * 1024 * 1024), TABLES_PER_TIER))
tiered_WA = 1 + tiered_levels   # much gentler: one rewrite per promotion

print(f"{'strategy':<18}{'write amp':>14}")
print(f"{'leveled':<18}{leveled_WA:>14}x")
print(f"{'size-tiered':<18}{tiered_WA:>14}x")
print()

# --- Read amplification --------------------------------------------------
# Without Bloom filters: touch memtable + every level that could hold the key.
# Leveled: L0 may have several overlapping tables (say 4), then 1 table/level.
L0_TABLES = 4
reads_no_bloom_leveled = 1 + L0_TABLES + (NUM_LEVELS - 1)  # memtable + L0 + one/level

# Size-tiered: each tier can have several tables, all potentially probed.
reads_no_bloom_tiered = 1 + TABLES_PER_TIER * tiered_levels

# With Bloom filters: each skipped SSTable only costs an expected FPR probe.
reads_bloom_leveled = 1 + L0_TABLES * BLOOM_FPR + (NUM_LEVELS - 1) * BLOOM_FPR + 1
#                          ^ false positives in levels that don't hold the key
#                          + 1 for the real hit at the correct level
reads_bloom_tiered  = 1 + (TABLES_PER_TIER * tiered_levels - 1) * BLOOM_FPR + 1

print(f"{'strategy':<18}{'reads/query':>16}{'+ bloom':>12}")
print(f"{'leveled':<18}{reads_no_bloom_leveled:>16}{reads_bloom_leveled:>12.2f}")
print(f"{'size-tiered':<18}{reads_no_bloom_tiered:>16}{reads_bloom_tiered:>12.2f}")
print()

# --- Space amplification -------------------------------------------------
# Leveled: at most (1 + 1/FANOUT + 1/FANOUT^2 + ...) ~ FANOUT/(FANOUT-1)
#          -> very close to 1.11x for fanout=10. Very low space amp.
leveled_space_amp = FANOUT / (FANOUT - 1)

# Size-tiered: worst case can be up to (num_tiers) copies of the same key
#              during the short window before tier promotion; steady-state
#              often quoted as ~2x.
tiered_space_amp  = 2.0

print(f"{'strategy':<18}{'space amp':>12}")
print(f"{'leveled':<18}{leveled_space_amp:>12.2f}x")
print(f"{'size-tiered':<18}{tiered_space_amp:>12.2f}x")

The picture is crisp: **leveled compaction pays ~6x more write amplification** to keep space amp near 1.1x and read amp at one-per-level. **Size-tiered inverts the triangle** — cheap writes, but multi-table probes per level and ~2x space amp. Bloom filters (at 1% FPR) essentially erase read amplification for negative lookups: you expect ~1 real SSTable touch plus a tiny fractional cost across all others.

## High-Level Taxonomy

```mermaid
graph TB
  Root["Log-Structured Storage"]
  Root --> LSM["Ordered LSM Tree<br/>memtable plus sorted SSTables<br/>merge-iteration on read"]
  Root --> Unordered["Unordered LSS<br/>records in insertion order<br/>no sort, no merge on read"]
  Root --> Stack["Log Stacking<br/>app log on FS log on SSD FTL<br/>multiple log layers"]

  LSM --> Leveled["Leveled compaction<br/>RocksDB"]
  LSM --> Tiered["Size-tiered compaction<br/>Cassandra"]
  LSM --> TimeWin["Time-window compaction<br/>time-series"]

  Unordered --> Bitcask["Bitcask<br/>in-memory keydir<br/>point queries only"]
  Unordered --> WiscKey["WiscKey<br/>key LSM plus vLog<br/>range queries via SSD parallelism"]

  Stack --> FTL["Flash Translation Layer<br/>hardware log"]
  Stack --> FSLog["Filesystem log<br/>F2FS, btrfs"]
  Stack --> LLAMA["LLAMA<br/>access-method-aware log"]
  Stack --> OpenCh["Open-Channel SSD<br/>bypass the FTL"]
```

## Deep Dive 1: LSM Tree Structure and Lifecycle

An LSM Tree splits its state across three kinds of components: a **current memtable** (mutable, in RAM, the write target), one or more **flushing memtables** (in RAM, frozen, in-flight to disk), and **disk-resident SSTables** (immutable sorted files). Every write appends to the WAL, then inserts into the current memtable. When the memtable crosses a size threshold, it is atomically **switched**: a fresh memtable takes new writes while the old one moves to the flushing state and is sorted+written out as a new SSTable. Once the SSTable lands, the matching WAL segment can be trimmed.

The chapter is emphatic about two-component vs. multicomponent designs. A **two-component LSM** has one big on-disk B-Tree and merges each memtable flush back into it in place (subtree by subtree). The write-amp from constantly merging into one giant structure is ruinous, so real systems use **multicomponent LSM**: every flush produces a brand-new SSTable, and background compaction merges a few SSTables into a larger one later. That decouples the flush rate from the merge rate — exactly the point of buffering.

The invariants during a flush are:

1. As soon as flush starts, all new writes go to the new memtable.
2. During the flush, both the disk-resident tables and the flushing memtable stay visible to reads.
3. Publishing the flushed SSTable and discarding the flushed memtable are atomic.
4. WAL truncation is coordinated with flush completion — never truncate a segment whose memtable has not yet landed on disk.

```mermaid
graph TB
  W["Write"] --> WAL["WAL append"]
  WAL --> CM["Current memtable<br/>sorted tree / skiplist"]

  CM -- "size threshold" --> Switch["Switch memtable"]
  Switch --> FM["Flushing memtable<br/>frozen, readable"]
  Switch --> CM2["New current memtable"]

  FM --> Flush["Flush to SSTable<br/>sequential write"]
  Flush --> SST0["SSTable L0"]
  SST0 -- "atomic publish" --> TableView["Table View"]
  FM -. "discard after publish" .-> X["trimmed"]
  WAL -. "trim after flush" .-> X

  SST0 --> Compact["Background compaction"]
  Compact --> SST1["SSTable L1"]
  SST1 --> Compact2["compact..."]
  Compact2 --> SSTn["SSTable Ln"]
```

**Why it matters**
- Writes never seek — memtable insert plus WAL append is one sequential IO.
- The memtable is the write coalescer: many updates to the same key collapse into one SSTable record on flush.
- Multicomponent (not two-component) is universal: decouples flush from merge so you can tune compaction independently.
- The entire durability story reduces to WAL + atomic SSTable publish + WAL trim.

## Deep Dive 2: Tombstones, Merge-Iteration, and Reconciliation

Because SSTables are immutable, you cannot delete a record by erasing it — an older SSTable might still hold the previous value. Delete it only from the memtable and the next flush resurrects the old value from disk. The fix is a **tombstone**: a special delete entry written into the memtable (and eventually into an SSTable) that shadows any prior value for that key at lower timestamps. A **range tombstone** (Cassandra) covers a key range instead of a single key.

A lookup has to pull from memtable + every candidate SSTable and reconcile. The mechanism is a **multiway merge-sort using a priority queue (min-heap)**: each iterator contributes its current head to the heap, the heap always yields the smallest key, and when a key pops you refill the heap from the iterator it came from. When two iterators contribute the same key you have a conflict — resolved by picking the record with the highest timestamp (or dropping everything if the winner is a tombstone).

Three critical properties:

- Per iterator, keys are sorted — so each iterator contributes at most one entry of a given key to the heap at a time.
- If two entries for the same key are in the heap, they *must* come from different iterators.
- Tombstones must survive compaction until they reach the bottommost level (RocksDB) or the GC grace period (Cassandra). Drop them too early and the shadowed record resurrects from a lower level.

```mermaid
graph LR
  IT0["Memtable iterator"] --> PQ["Min-heap priority queue<br/>holds one head per iterator"]
  IT1["SSTable 1 iterator"] --> PQ
  IT2["SSTable 2 iterator"] --> PQ
  IT3["SSTable 3 iterator"] --> PQ

  PQ -- "pop min key" --> R["Reconciler<br/>pick highest timestamp<br/>drop tombstoned keys"]
  R --> Out["Sorted output<br/>client or compaction writer"]
  R -- "refill from source iterator" --> PQ
```

Let's follow the exact example the chapter walks through, then extend it with a tombstone.

In [ ]:
# Multiway merge with timestamp reconciliation and tombstone shadowing.
import heapq

TOMBSTONE = object()  # sentinel

# Two sorted iterators (key, value, timestamp). Higher timestamp wins.
# Iterator 1: {k2:v1} {k4:v2}
# Iterator 2: {k1:v3} {k2:v4} {k3:v5}
# The chapter example: Iterator 2's k2:v4 is newer than Iterator 1's k2:v1.
iter1 = iter([("k2", "v1", 1), ("k4", "v2", 1)])
iter2 = iter([("k1", "v3", 2), ("k2", "v4", 2), ("k3", "v5", 2)])

def multiway_merge(iterators):
    """Yield (key, value, ts) in sorted order, reconciling duplicates."""
    # Heap entries: (key, source_index, value, ts). We include source_index so
    # equal keys never collide on (value, ts) comparison.
    heap = []
    sources = [iter(it) for it in iterators]
    for i, src in enumerate(sources):
        nxt = next(src, None)
        if nxt is not None:
            k, v, ts = nxt
            heapq.heappush(heap, (k, i, v, ts))

    while heap:
        # Pull all entries matching the smallest key out of the heap.
        smallest_key = heap[0][0]
        candidates = []
        while heap and heap[0][0] == smallest_key:
            k, i, v, ts = heapq.heappop(heap)
            candidates.append((i, v, ts))
            # Refill from source i.
            nxt = next(sources[i], None)
            if nxt is not None:
                nk, nv, nts = nxt
                heapq.heappush(heap, (nk, i, nv, nts))

        # Reconciliation: highest timestamp wins. Tombstones shadow real records.
        winner = max(candidates, key=lambda c: c[2])
        _, v, ts = winner
        if v is TOMBSTONE:
            continue  # shadowed / deleted; emit nothing
        yield (smallest_key, v, ts)

print("Chapter example (no tombstones):")
for entry in multiway_merge([iter1, iter2]):
    print(" ", entry)
print()
# Expected: (k1,v3,2) (k2,v4,2) (k3,v5,2) (k4,v2,1)

# --- Extension: add a tombstone for k3 in a newer iterator -----------------
# Iterator 1: {k2:v1@ts1} {k4:v2@ts1}
# Iterator 2: {k1:v3@ts2} {k2:v4@ts2} {k3:v5@ts2}
# Iterator 3: {k3:<tombstone>@ts3}  -- newest; deletes k3
iter1 = iter([("k2", "v1", 1), ("k4", "v2", 1)])
iter2 = iter([("k1", "v3", 2), ("k2", "v4", 2), ("k3", "v5", 2)])
iter3 = iter([("k3", TOMBSTONE, 3)])

print("With tombstone on k3:")
for entry in multiway_merge([iter1, iter2, iter3]):
    print(" ", entry)
# k3 disappears from output: the ts=3 tombstone shadows the ts=2 value.

**Why it matters**
- Merge-iteration is the only way to read an LSM Tree — every point query and range scan walks a heap over the same iterator shapes.
- Reconciliation by timestamp is what lets writes be idempotent and commutative — updates and deletes can arrive in any order across SSTables.
- Tombstones are not bookkeeping; they are first-class records that must survive compaction to prevent data resurrection.
- Range tombstones encode bulk deletes cheaply but make reconciliation boundary-aware — overlapping ranges across SSTables need careful inclusive/exclusive handling.

## Deep Dive 3: Compaction Strategies

Compaction is the janitor that keeps LSM lookups from degenerating into probes of hundreds of files. It picks several SSTables, merge-iterates them, reconciles duplicates and tombstones, and writes a fresh SSTable out. Because reads and compaction both rely on the same merge-iteration machinery, the *strategy* — which SSTables to pick and when — dominates the write/read/space amp trade.

**Leveled compaction (RocksDB)** partitions SSTables into numbered levels with exponentially growing target sizes (10x per level is typical). **L0 tables can have overlapping key ranges** (they are just raw memtable flushes), but **L1 and deeper levels are partitioned into non-overlapping key ranges across their SSTables**. So a point lookup probes at most one SSTable per level (plus however many L0 tables can overlap). When a level exceeds its budget, a table from it is merged with every overlapping table on the next level to produce new non-overlapping tables on the next level. Every record gets rewritten ~10x on its way down — high write amp, low space amp (~1.11x for fanout 10), predictable read amp.

**Size-tiered compaction (Cassandra default)** groups tables by size class: when you have 4 tables in a size bucket, merge them into one table that lands in the next bucket up. Much gentler write amp (each record is rewritten once per tier promotion), but reads may have to probe all 4 tables per tier, and space amp climbs (~2x) because multiple versions of the same key can sit in peer tables within a tier. The canonical pain point is **table starvation**: if compacted tables stay small (lots of tombstones shadowed records), higher tiers never fill up, tombstones accumulate, and read cost explodes. Operators periodically **force compaction** to drain tombstones.

**Time-window compaction** groups SSTables by write-timestamp window (one SSTable per hour, say). When a window passes its TTL, *the whole SSTable is dropped* — no merge, no scan. This is the ideal strategy for time-series workloads with uniform TTLs.

```mermaid
graph TB
  subgraph Leveled["Leveled Compaction"]
    L0["L0 SSTables<br/>overlapping ranges<br/>from memtable flushes"]
    L1["L1 SSTables<br/>non-overlapping<br/>10x memtable"]
    L2["L2 SSTables<br/>non-overlapping<br/>100x memtable"]
    L3["L3 SSTables<br/>non-overlapping<br/>1000x memtable"]
    L0 -- "merge overlapping L1" --> L1
    L1 -- "merge overlapping L2" --> L2
    L2 -- "merge overlapping L3" --> L3
  end

  subgraph Tiered["Size-Tiered Compaction"]
    T0["Tier 0: 4 small tables"]
    T1["Tier 1: 4 medium tables"]
    T2["Tier 2: 4 large tables"]
    T0 -- "bucket full: merge 4 into 1" --> T1
    T1 -- "bucket full: merge 4 into 1" --> T2
  end

  subgraph TimeWin["Time-Window Compaction"]
    TW0["Hour 00 SSTable"]
    TW1["Hour 01 SSTable"]
    TW2["Hour 02 SSTable"]
    TW3["Hour 03 SSTable"]
    Drop["Window past TTL<br/>DROP FILE"]
    TW0 -- "TTL expired" --> Drop
  end
```

| Strategy | Write Amp | Read Amp | Space Amp | Best For |
|---|---|---|---|---|
| Leveled | High (~fanout per level) | Low (1 per level + L0) | ~1.1x | Point-heavy OLTP, SSD-resident, RocksDB-style |
| Size-Tiered | Low (log_k N) | Medium-high (k per tier) | ~2x | Write-heavy, tombstone-light, Cassandra-style |
| Time-Window | Very low (drop, no merge) | Depends on active windows | Depends | Time-series, TTL workloads |

**Why it matters**
- Compaction strategy is the single biggest knob for tuning the RUM trade in an LSM.
- L0 is inevitably a read-amp hotspot because flushed tables overlap — that's why L0 usually has separate, tighter limits.
- Tombstones demand compaction to make progress; size-tiered's starvation problem is a real operational failure mode.
- Time-window only works when you can afford to keep whole files homogeneous in age — one out-of-band write ruins it.

## Deep Dive 4: RUM Conjecture and the Three Amplifications

The **RUM Conjecture** [Athanassoulis et al., 2016] states: a storage engine can optimize at most two of **R**ead overhead, **U**pdate overhead, and **M**emory (space) overhead; improving two worsens the third. It is a lens for explaining why there is no free lunch across B-Trees, LSM Trees, and their variants.

- **B-Trees** optimize R and (historically) M at the expense of U. Lookups are cheap (one page per level) and on-disk layout is dense-ish, but every update pays a full-page rewrite and slack-for-future-inserts inflates M somewhat.
- **LSM (leveled)** pushes U way down (memtable buffering + append-only SSTables) and keeps M tight (space amp ~1.1x), but pays R across multiple SSTables — mitigated by Bloom filters.
- **LSM (size-tiered)** pushes U even lower but raises both R (more tables per tier) and M (up to ~2x).
- **Bitcask** aces both R and U (in-memory keydir → one disk seek for a point query; append-only writes) but blows up M — every key sits in RAM forever.

The LSM family exposes all three amplifications as explicit tuning dials:

- **Read amplification** — how many SSTables a lookup probes. Knobs: Bloom filters, compaction strategy, level count.
- **Write amplification** — how many times a record is rewritten from memtable to bottommost level. Knob: compaction aggressiveness.
- **Space amplification** — how many redundant versions of the same key coexist. Knob: compaction frequency + tombstone retention.

```mermaid
graph TB
  subgraph RUM["RUM Triangle"]
    R["Read overhead"]
    U["Update overhead"]
    M["Memory / space overhead"]
    R --- U
    U --- M
    M --- R
  end

  BTree["B-Tree<br/>low R, high U, low-mid M"]
  LSMLv["LSM leveled<br/>mid R, mid U, very low M"]
  LSMSz["LSM size-tiered<br/>high R, low U, mid M"]
  Bit["Bitcask<br/>very low R+U, very high M<br/>all keys in RAM"]

  BTree -.-> R
  LSMLv -.-> U
  LSMSz -.-> U
  Bit -.-> R
  Bit -.-> U
```

| System | Read Amp | Write Amp | Space Amp | Memory |
|---|---|---|---|---|
| B-Tree (in-place) | O(log n) one page/level | 1 full page per update | ~1.4x (fill slack) | Page cache |
| LSM (leveled) | ~levels + L0 (w/ Bloom: ~1) | ~fanout x levels | ~1.1x | Memtable + Bloom + index |
| LSM (size-tiered) | k x tiers (w/ Bloom: ~1) | ~log_k n | ~2x | Memtable + Bloom + index |
| Bitcask | 1 seek | 1 append | ~1x on disk | All keys in RAM |

**Why it matters**
- Every LSM tuning decision maps onto the RUM triangle; there is no "just tune it" escape hatch.
- Bloom filters are the classical way to buy R back after U and M have been optimized.
- Aggressive compaction trades W amp for R and S amp; lazy compaction inverts it. Operators flip this dial by workload.
- Bitcask is the RUM endpoint that trades all of M: point-read optimal, but memory grows with cardinality.

## Deep Dive 5: Read-Path Optimizations — SSTables, Bloom Filters, Skiplists

The LSM read path has three layers of optimization, one per read-time bottleneck.

### Sorted String Tables (SSTables)

SSTables are the disk-resident format: data records sorted by key, laid out in a **data file**, with a companion **index file** mapping keys to offsets in the data file. The index is usually a B-Tree (for log-time lookup) or a hashtable (constant-time; still supports range scans by finding the starting key and then sequentially reading the sorted data file). The data file is **compressed page-wise**, with a side table mapping logical page offsets to physical compressed offsets (compression breaks page alignment). During compaction, the data file is read purely sequentially — the index is rebuilt for the output, but isn't needed for reading inputs.

### Bloom filters

Every SSTable has an associated Bloom filter sized to the SSTable's key count. Before opening an SSTable's index file, the read path tests the filter. A **negative** is definitive — skip the SSTable entirely. A **positive** is probabilistic — could be a real hit or a false positive at rate `p`. The filter is a bit array of size `m` plus `k` hash functions; `k_opt = (m/n) ln 2` for optimal `k` given `m` bits and `n` keys.

### Skiplists

Memtables need concurrent-safe sorted inserts. Red-black / AVL trees work but require rotations and fine-grained locks. **Skiplists** sidestep both: each node has a probabilistic tower of forward pointers (each level has exponentially fewer nodes); insert picks a random height geometric-distributed. Lookups descend from the highest level, following forward pointers until the next key exceeds the target, then drop a level. Concurrency is handled with per-node CAS on a `fully_linked` flag. Skiplists are slightly less cache-friendly than in-memory B-Trees but massively simpler and lock-free by construction. Cassandra uses them for secondary index memtables; WiredTiger uses them for WiredTiger's own per-node update buffers.

```mermaid
graph TB
  subgraph SST["SSTable layout"]
    IDX["Index file<br/>B-Tree or hashtable<br/>key -> offset"]
    DAT["Data file<br/>sorted key/value pairs<br/>page-wise compressed"]
    BLM["Bloom filter<br/>bit array plus k hashes<br/>sidecar"]
    IDX --> DAT
    BLM -. "test before opening" .-> IDX
  end

  subgraph Skip["Skiplist memtable"]
    H["head"]
    H --> L3a["L3 node"]
    L3a --> L3b["L3 node"]
    H --> L2a["L2 node"]
    L2a --> L2b["L2 node"]
    L2b --> L2c["L2 node"]
    H --> L1a["L1 node"]
    L1a --> L1b["L1 node"]
    L1b --> L1c["L1 node"]
    L1c --> L1d["L1 node"]
  end

  subgraph Bloom["Bloom filter"]
    BA["16-bit array"]
    K1["key1 hashes to 3, 5, 10"]
    K2["key2 hashes to 5, 8, 14"]
    K3["key3 hashes to 3, 10, 14<br/>all bits set -> false positive"]
    K4["key4 hashes to 5, 9, 15<br/>bit 9 is 0 -> definite miss"]
  end
```

A concrete Bloom filter worked example, reproducing the chapter's `key1`, `key2`, `key3`, `key4` setup and then sweeping `k` and `m` to show how FPR depends on both.

In [ ]:
# Bloom filter: chapter example + FPR sensitivity sweep.
import math

# --- Chapter example: 16-bit array, 3 hash functions -----------------------
m = 16
bits = [0] * m

# These are the exact bit positions from the chapter.
key1_bits = [3, 5, 10]
key2_bits = [5, 8, 14]

for b in key1_bits:
    bits[b] = 1
for b in key2_bits:
    bits[b] = 1

def probe(name, positions):
    vals = [bits[p] for p in positions]
    verdict = "MIGHT BE PRESENT" if all(vals) else "DEFINITELY ABSENT"
    print(f"  {name}: positions={positions}, bits={vals} -> {verdict}")

print("Bit array after inserting key1, key2:")
print(f"  {bits}\n")

print("Queries:")
probe("key1 (inserted)       ", key1_bits)
probe("key2 (inserted)       ", key2_bits)
probe("key3 (not inserted)   ", [3, 10, 14])  # false positive
probe("key4 (not inserted)   ", [5, 9, 15])   # definite miss
print()

# --- FPR formula: (1 - exp(-k*n/m))^k --------------------------------------
def bloom_fpr(m, n, k):
    return (1.0 - math.exp(-k * n / m)) ** k

# Sweep: for n=1000 keys and varying m (bits), find optimal k and its FPR.
n = 1000
print(f"FPR sweep for n={n} keys")
print(f"{'m (bits)':>10}{'bits/key':>10}{'k_opt':>8}{'FPR':>12}")
for m_bits in (1000, 5000, 10_000, 50_000, 100_000):
    k_opt = max(1, round((m_bits / n) * math.log(2)))
    fpr = bloom_fpr(m_bits, n, k_opt)
    print(f"{m_bits:>10}{m_bits/n:>10.1f}{k_opt:>8}{fpr:>12.4%}")
print()

# --- Practical sizing: what FPR do I get for 10 bits/key and k=7? ---------
print("Typical LSM sizing: 10 bits/key, k=7 -> FPR =",
      f"{bloom_fpr(10 * n, n, 7):.4%}")
# ~0.82%, the standard RocksDB default.

**Why it matters**
- SSTable immutability is what makes page-wise compression, fixed Bloom filter sizing, and lock-free reads all possible.
- Bloom filters convert "multi-level read amp" into "1 true hit + negligible FPR cost per level" — without them, LSM reads would be economically untenable.
- Optimal Bloom sizing `k = (m/n) ln 2` is a hard theorem; LSMs usually ship with `10 bits/key, k=7, ~0.8% FPR` as a default.
- Skiplists dominate for the memtable because they are simple, probabilistically balanced, and CAS-friendly — all properties trees struggle to provide simultaneously.

## Deep Dive 6: Unordered Log-Structured Storage — Bitcask and WiscKey

Sorted LSM Trees pay a lot of work to keep data in key order: sort on flush, merge on compaction, partition on leveled compaction. **Unordered** LSS gives that up. Records go to disk in **insertion order** inside append-only logfiles. There is no memtable (Bitcask) or no value sorting (WiscKey). The price is that you cannot do merge-iteration for range scans directly — you must find records some other way.

### Bitcask (Riak)

- **Writes** append the (key, value, timestamp, size) record directly to the active logfile.
- **Keydir** is an in-memory hashmap `key -> (file_id, offset, size, ts)` pointing at the latest record for each key. Updated on every write.
- **Reads** look up the keydir (O(1) hash probe), seek to the recorded offset, and read the record. Exactly **one disk seek per point query**, no merging.
- **Compaction** reads logfiles sequentially, keeps only records whose keydir entry points at them (i.e., the live records), writes a new file, and updates the keydir.

The trade is stark: **zero range-query support** (neither keydir nor logfiles are sorted), plus **all keys must fit in RAM** (and must be rebuilt from logfiles on startup, which is slow for huge datasets). But point-query performance is essentially optimal.

### WiscKey

WiscKey splits the difference: **keys live in a sorted LSM Tree; values live in an append-only unsorted vLog**. The LSM Tree stores `(key, vLog_offset)`; compaction only rewrites keys (tiny), which is massively cheaper than rewriting full records on every compaction. A vLog has a **head** (where new records are appended) and a **tail** (where GC advances as it reclaims space). GC reads from the tail, consults the key LSM Tree to find which values are still live, and rewrites the live ones at the head. Range scans read sorted keys from the LSM, then issue random IOs to vLog offsets — painful on spinning disk but mitigated on SSD by **internal parallelism** (issuing many reads at once saturates NAND channels).

```mermaid
graph TB
  subgraph BC["Bitcask"]
    KD["In-memory keydir<br/>hashmap key to (fid, off)"]
    LF1["Logfile 1 (closed)<br/>append-only"]
    LF2["Logfile 2 (active)<br/>append-only"]
    KD -- "read: 1 seek" --> LF1
    KD -- "read: 1 seek" --> LF2
    W["Write"] --> LF2
    W --> KD
  end

  subgraph WK["WiscKey"]
    KLSM["Key LSM Tree<br/>sorted keys plus vLog offsets"]
    VLog["vLog<br/>unsorted value records<br/>head and tail pointers"]
    KLSM -- "value_ptr" --> VLog
    W2["Write"] --> VLog
    W2 --> KLSM
    GC["GC reads from tail<br/>consults key LSM<br/>rewrites live records at head"]
    GC --> VLog
  end
```

| System | Point Query | Range Query | Memory Footprint | Startup Cost | GC Complexity |
|---|---|---|---|---|---|
| Classical sorted LSM | Multi-table probe + Bloom | Merge-iterate SSTables | Memtable + Bloom + index | Fast (indexes on disk) | Standard compaction |
| Bitcask | 1 seek (hashmap hit) | Not supported | All keys in RAM | Slow (rebuild keydir) | Sequential log merge |
| WiscKey | LSM lookup + 1 vLog seek | LSM scan + many parallel vLog seeks | Normal LSM | Fast | Needs key-LSM consultation |

**Why it matters**
- Bitcask shows the extreme of the RUM trade: near-optimal R and U, catastrophic M.
- WiscKey is the key insight that keys and values don't need the same treatment — sort what you query on, stream what you store.
- Key/value separation scales with value size: for 10 KB values, WiscKey's compaction surface is ~100x smaller than a classical LSM with the same keys.
- SSD internal parallelism is what makes WiscKey's random-IO range scans viable — would not work on spinning disks.

## Deep Dive 7: Log Stacking — FTL, Filesystem Logging, LLAMA, Open-Channel SSDs

An LSM Tree rarely runs on bare metal. The typical stack is **app LSM on top of a log-structured filesystem on top of the SSD's flash translation layer (FTL)**. Each layer independently buffers writes, flushes segments, and garbage-collects old segments. This is transparent to the layer above but causes three problems:

1. **Duplicated GC work.** Every layer re-runs its own garbage collection, unaware that the layer above has already logically reclaimed the same data. The FTL doesn't know the LSM already discarded an SSTable; the filesystem doesn't know the FTL is about to erase a block.
2. **Misaligned segments.** LSM segments, filesystem extents, and SSD erase blocks all have different sizes. Discarding an LSM segment can slice across multiple FS extents, which in turn slice across SSD erase blocks. Small app-level writes cause massive hardware-level rewrites (write amplification multiplied across layers).
3. **Interleaved write streams.** An LSM does WAL writes in parallel with SSTable writes; the FS sees two sequential streams but writes them interleaved into FS log blocks; the FTL sees one scrambled stream that no longer corresponds to any single logical sequence.

The FTL itself is a log-structured subsystem: SSDs are flash, flash can only be written to erased pages, and pages can only be erased in groups of 64-512 called **blocks**. When the FTL runs out of free pages, it picks a block, relocates its live pages elsewhere, and erases it. This is wear-leveling and GC in one. If the OS and the app are also doing this, you have three layers doing the same work on different granularities.

```mermaid
graph TB
  subgraph App["App layer: LSM Tree"]
    AS1["SSTable segment A"]
    AS2["SSTable segment B"]
    AS3["SSTable segment C"]
  end

  subgraph FS["Filesystem log"]
    FS1["FS segment 1<br/>holds half of A plus all WAL"]
    FS2["FS segment 2<br/>holds half of A plus half of B"]
    FS3["FS segment 3<br/>holds half of B plus all of C"]
  end

  subgraph SSD["SSD FTL: erase blocks"]
    EB1["Erase block 1<br/>holds FS1 plus half of FS2"]
    EB2["Erase block 2<br/>holds half of FS2 plus FS3"]
  end

  AS1 --> FS1
  AS1 --> FS2
  AS2 --> FS2
  AS2 --> FS3
  AS3 --> FS3
  FS1 --> EB1
  FS2 --> EB1
  FS2 --> EB2
  FS3 --> EB2

  Note["Discarding SSTable A<br/>invalidates parts of FS1 and FS2<br/>invalidates parts of EB1 and EB2<br/>all three layers must independently GC"]
```

The chapter offers three escape routes:

### LLAMA (access-method aware log)

LLAMA is the log-structured subsystem under the Bw-Tree. Its key idea: **the log is aware that it stores Bw-Tree delta chains**. When LSS does GC, instead of blindly rewriting delta nodes contiguously (which would still require the reader to walk the chain), it **consolidates** the chain into a fresh base page with all deltas pre-applied. It also drops canceling deltas (insert+delete = nothing). This piggybacks logical Bw-Tree consolidation on physical LSS GC — one pass, not two. The broader lesson: a log layer that knows the *shape* of its data can eliminate a huge class of redundant work.

### Open-Channel SSDs (LOCS, LightNVM, SDF)

Skip the FTL. Open-Channel SSDs expose raw flash channels, erase blocks, and wear-leveling state to the application. The app (e.g. LOCS, an LSM directly on Open-Channel SSD) handles page allocation, GC, and wear-leveling itself, aligned to the LSM's own segment sizes. Software Defined Flash takes it further with **asymmetric read/write units** — small read units (page-sized) and large write units (erase-block-sized) — perfectly matched to log-structured workloads. This is the `O_DIRECT` philosophy applied to SSDs: more control, more code, less accidental overhead.

### Mindful stacking: the general principle

When you must stack logs, **align segment sizes across layers** (LSM segment = FS extent = SSD erase block), and **keep workloads with different access patterns on separate devices** (WAL on one SSD, SSTables on another). Don't pretend the layers are independent.

**Why it matters**
- Every production LSM runs on stacked logs; the default arrangement is quietly costing you 2-5x write amplification on top of whatever LSM-level write amp you already have.
- LLAMA is the archetype of "the log knows what it stores" — a principle that applies to any log-structured subsystem under another one.
- Open-Channel SSDs trade simplicity for control; choose them when LSS is your bottleneck and you can afford the engineering budget.
- The simplest practical win is segment-size alignment; misalignment is the number one cause of inflated hardware-level write amp.

## Trade-offs and Alternatives

| Dimension | B-Tree (in-place) | LSM (leveled) | LSM (size-tiered) | Bitcask | WiscKey |
|---|---|---|---|---|---|
| Write path | Locate + rewrite page | Memtable append + WAL | Memtable append + WAL | Append to logfile | Append to vLog + LSM insert |
| Read path | 1 page per level | Memtable + 1 per level + Bloom | Memtable + k per tier + Bloom | Keydir + 1 seek | LSM + 1 vLog seek |
| Write amp | 1 page per update | ~fanout x levels | ~log_k n | ~1 (no compaction rewrites) | Low (keys only) |
| Space amp | ~1.4x (slack) | ~1.1x | ~2x | ~1x on disk | Moderate |
| Memory | Page cache | Memtable + Bloom + index | Memtable + Bloom + index | All keys in RAM | Memtable + Bloom + index |
| Range scans | Excellent | Good (merge iteration) | Good (merge iteration) | Not supported | Moderate (SSD parallel) |
| Tombstones | N/A | Required; must reach bottom | Required; starvation risk | Keydir delete | Required in key LSM |
| Representative | PostgreSQL, InnoDB | RocksDB, LevelDB | Cassandra | Riak | WiscKey, Badger |
| Best fit | Read-heavy, transactional | Mixed R/W, point-heavy | Write-heavy, tombstone-light | Point queries, small keys | Large values, mixed R/W |

## Key Takeaways

1. **Immutability turns random writes into sequential ones.** The LSM Tree is the canonical expression: buffer in memtable, WAL for durability, flush immutable SSTables, compact lazily. Every other design choice in the chapter follows from this.
2. **Multicomponent beats two-component.** Merging every flush into one giant disk B-Tree kills write amplification; decoupling flush from merge (many SSTables, lazy compaction) is why every real LSM is multicomponent.
3. **Tombstones are first-class records**, not bookkeeping. They must survive compaction until the bottommost level (RocksDB) or GC grace (Cassandra), otherwise shadowed records resurrect.
4. **Merge-iteration with a min-heap** is the universal LSM read primitive — one algorithm serves point queries, range scans, and compaction.
5. **Compaction strategy is the RUM dial.** Leveled minimizes space at the cost of write amp; size-tiered inverts the triangle; time-window drops whole files for TTL workloads. Pick one per workload.
6. **Bloom filters are what make LSM reads viable.** Standard sizing (10 bits/key, k=7, ~0.8% FPR) eliminates nearly all false probes; without them, per-level read amp would be catastrophic.
7. **Skiplists rather than trees for the memtable.** Simple, probabilistic, CAS-friendly — no rotations, no fine-grained locks.
8. **Unordered LSS trades sort for RAM.** Bitcask wins point queries and loses range scans and memory; WiscKey splits the difference by keeping keys in an LSM and values in an unsorted vLog.
9. **The RUM Conjecture is a law, not a guideline.** You can optimize at most two of Read, Update, Memory. Every LSM tuning choice is a move in this triangle.
10. **Log stacking is the hidden tax.** App LSM on FS log on SSD FTL causes duplicated GC, misaligned segments, and interleaved streams. LLAMA fixes it by making the log access-method aware; Open-Channel SSDs let you bypass the FTL entirely.

## See Also

- [[01-lsm-tree-structure]]
- [[02-tombstones-and-merge-reconciliation]]
- [[03-compaction-strategies]]
- [[04-rum-conjecture-and-amplification]]
- [[05-bloom-filters-and-skiplists]]
- [[06-unordered-log-structured-storage]]
- [[07-log-stacking-and-hardware-awareness]]